# 15 — Why Rankings Invert, Replication of the Update Sweep, and a Durability Control

Four additions, on the same corpora, caps and splits as notebooks 10 to 14.

Inversion mechanism. Notebook 12 found that half of all transferred rankings are inverted and notebook 11 that inversion is not explained by prior shift, but neither asked why it happens. Here each feature's point-biserial correlation with the label is computed per corpus, features whose sign reverses between a pair are identified, and an importance-weighted sign-agreement score is formed as a candidate predictor of whether a transferred ranking survives. The causal claim is then tested directly: source models are refit with the sign-reversed features removed, against a control that removes an equal number of randomly chosen features, and the zero-shot transfer is re-measured.

Replication. The update-size sweep of notebook 14 ran at one seed and overturned a conclusion of notebook 13, so it is repeated at seeds 43 and 44.

Durability control. Notebook 14's family holdout removes one attack family entirely, which changes both novelty and buffer composition. The control here retains the family at a tenth of its normal share, holding buffer size fixed, so that recall on that family separates an unseen attack type from a less diverse buffer.

Results append to fc_results_v6.csv (per-cell) and fc_features.csv (per-feature correlations) with resume-skip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings, copy, zlib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seed          = 42,
    rep_seeds     = [43, 44],
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    ft_budgets    = [0.001, 0.01],
    ft_mult       = [1, 3, 10],
    hold_budget   = 0.001,
    hold_keep     = 0.1,
    hold_min_rows = 2_000,
    hold_max_share= 0.60,
    max_families  = 3,
    tau           = 0.05,
    corr_sample   = 400_000,
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
    fpr_caps      = [0.01, 0.001],
)
MODELS = ['rf', 'lgbm', 'mlp']
V6_CSV = f'{RESULT}/fc_results_v6.csv'
FEA_CSV = f'{RESULT}/fc_features.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train', 'held_family',
        'keep_share', 'n_features', 'n_dropped', 'mcc', 'macro_f1', 'fp_rate', 'auprc_macro',
        'mcc_best_thr', 'orient', 'tpr_at_fpr01_buffer', 'tpr_heldout_family', 'fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
import lightgbm as lgb

# ---------- fine-tuning: keep the source model, adapt it on the buffer ----------
def finetune(model, name, Xb, yb, seed, add_trees=100, ft_epochs=30):
    """Family-specific continuation of a fitted source model on the buffer.
    RF: warm_start adds `add_trees` grown on the buffer, source trees retained.
    LightGBM: boosting continues from the source booster via init_model.
    MLP: SGD continues from the source weights, with the buffer transformed by the
    SOURCE scaler; refitting the pipeline would refit the scaler on the buffer and
    apply the source weights to differently-scaled inputs.
    A single-class buffer offers nothing to adapt to, so the source model is returned."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return model
    if name == 'rf':
        model.set_params(warm_start=True, n_estimators=model.n_estimators + add_trees)
        model.fit(Xb, yb)
        return model
    if name == 'lgbm':
        m2 = lgb.LGBMClassifier(n_estimators=add_trees, random_state=seed, n_jobs=-1, verbosity=-1)
        m2.fit(Xb, yb, init_model=model.booster_)
        return m2
    scaler = model.named_steps['scaler']
    clf = model.named_steps['clf']
    # sklearn leaves best_loss_ unset when the source fit used early stopping; the
    # non-early-stopping update path dereferences it, so it is restored here.
    if getattr(clf, 'best_loss_', None) is None:
        curve = getattr(clf, 'loss_curve_', None)
        clf.best_loss_ = float(curve[-1]) if curve else np.inf
    clf._no_improvement_count = 0
    clf.set_params(warm_start=True, max_iter=ft_epochs, early_stopping=False)
    clf.fit(scaler.transform(Xb), yb)
    return model


# ---------- importance weighting: reweight source rows towards the target ----------
def importance_weights(Xs, Xb, seed, clip=(0.05, 20.0), n_src=20000):
    """Domain discriminator p(target|x); source weight = p/(1-p).
    The discriminator is trained on a class-balanced sample and with balanced class
    weights, because an unbalanced source/buffer ratio otherwise drives every source
    probability to zero and collapses the weights onto the clip floor."""
    rng = np.random.default_rng(seed)
    n = min(n_src, len(Xs), max(len(Xb), 200))
    idx = rng.choice(len(Xs), size=min(n_src, len(Xs)), replace=False)
    src_d = np.asarray(Xs)[rng.choice(len(Xs), size=n, replace=False)]
    tgt_d = np.asarray(Xb)
    if len(tgt_d) < n:
        tgt_d = tgt_d[rng.choice(len(tgt_d), size=n, replace=True)]
    Xd = np.vstack([src_d, tgt_d])
    yd = np.r_[np.zeros(len(src_d)), np.ones(len(tgt_d))]
    mu, sd = Xd.mean(0), Xd.std(0) + 1e-9
    lr = LogisticRegression(max_iter=300, C=1.0, class_weight='balanced')
    lr.fit((Xd - mu) / sd, yd)
    p = lr.predict_proba((np.asarray(Xs) - mu) / sd)[:, 1]
    w = p / np.clip(1 - p, 1e-6, None)
    w = np.clip(w, *clip)
    return w / w.mean()


# ---------- selector estimator that does not collapse on tiny buffers ----------
def cv_estimate_v2(make_model_fn, Xb, yb, seed):
    """Stratified CV with folds adapted to the rarest class; falls back to a
    single stratified holdout when the minority class has 2 rows, and to the
    buffer-resubstitution estimate when it has 1. Never returns 0 by default."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return 0.0, 'single_class'
    nmin = int(np.bincount(yb).min())
    if nmin >= 3:
        k = min(3, nmin)
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        return (float(np.mean(out)), f'cv{k}') if out else (0.0, 'cv_empty')
    if nmin == 2:
        skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        if out and max(out) > 0:
            return float(np.mean(out)), 'holdout2'
        m = make_model_fn(len(Xb)); m.fit(Xb, yb)
        return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub2'
    m = make_model_fn(len(Xb)); m.fit(Xb, yb)
    return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub'

# ---------- iterative active learning: the retrained model drives the queries ----------
def entropy(p):
    p = np.clip(np.asarray(p), 1e-9, 1 - 1e-9)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def al_iterative(make_model_fn, pool, features, clean_fn, k_total, rounds, seed, stratify_col='Attack'):
    """Round 1 is a stratified random seed set; later rounds query the highest-entropy
    pool rows under the model retrained on what has been labelled so far."""
    per = max(1, k_total // rounds)
    rng_state = seed
    first = pool.groupby(stratify_col, group_keys=False).apply(
        lambda g: g.sample(n=max(1, int(round(len(g) * per / len(pool)))), random_state=rng_state))
    chosen = list(first.index[:per])
    for _ in range(rounds - 1):
        lab = pool.loc[chosen]
        Xl, med = clean_fn(lab, features)
        yl = lab['Label'].values
        if len(np.unique(yl)) < 2:
            rest = pool.index.difference(chosen)
            chosen += list(pd.Index(rest)[:per]); continue
        m = make_model_fn(len(Xl)); m.fit(Xl, yl)
        rest = pool.index.difference(chosen)
        Xr, _ = clean_fn(pool.loc[rest], features, medians=med)
        e = entropy(m.predict_proba(Xr)[:, 1])
        chosen += list(pd.Index(rest)[np.argsort(-e)[:per]])
    return pool.loc[chosen[:k_total]]

# ---------- INSOMNIA-style: pseudo-labelled pool plus uncertainty-queried labels ----------
def insomnia_buffer(source_model, pool, features, clean_fn, med_s, k, seed, conf=0.9, cap=50000):
    """Approximates INSOMNIA: the frozen source model pseudo-labels confident pool rows,
    and the analyst budget is spent on the rows it is least certain about."""
    Xp, _ = clean_fn(pool, features, medians=med_s)
    p = source_model.predict_proba(Xp)[:, 1]
    q = pool.index[np.argsort(-entropy(p))[:k]]
    conf_mask = (p >= conf) | (p <= 1 - conf)
    conf_idx = pool.index[conf_mask].difference(q)
    if len(conf_idx) > cap:
        conf_idx = pd.Index(np.random.default_rng(seed).choice(conf_idx, cap, replace=False))
    pseudo = pool.loc[conf_idx].copy()
    pseudo['Label'] = (p[pool.index.get_indexer(conf_idx)] >= 0.5).astype(int)
    return pool.loc[q], pseudo

In [ ]:
import numpy as np, pandas as pd

def tpr_at_fpr(y, p, target_fpr):
    """Highest TPR attainable at or below target_fpr, with the threshold that attains it.
    Thresholds are evaluated at every distinct score, so the result is exact."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind='mergesort'); ys, ps = y[order], p[order]
    P = int(ys.sum()); N = len(ys) - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last], fp[last], ps[last]
    ok = (fp / N) <= target_fpr
    if not ok.any():
        return 0.0, float(cuts[0]) + 1e-12
    i = int(np.argmax(np.where(ok, tp, -1)))
    return float(tp[i] / P), float(cuts[i])

def threshold_for_fpr_on_buffer(y_buf, p_buf, target_fpr):
    """Operating point an analyst could actually set: chosen on the labelled buffer only."""
    _, thr = tpr_at_fpr(y_buf, p_buf, target_fpr)
    return thr

def realised_at_threshold(y, p, thr):
    """TPR and FPR on the evaluation set at an externally chosen threshold."""
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    P = int(y.sum()); N = len(y) - P
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return (tp / P if P else np.nan), (fp / N if N else np.nan)

def family_holdout_buffer(train_full, held_family, frac, seed, min_per_group=1):
    """Stratified buffer drawn only from families other than held_family, so the retrained
    model has never seen that attack type. Size matches the ordinary buffer at this budget."""
    pool = train_full[train_full['Attack'] != held_family]
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in pool.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * k / len(pool)))))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def eligible_families(df, min_rows=2000, max_share=0.60):
    """Attack families large enough to matter but not so dominant that holding one out
    leaves nothing to train on."""
    v = df.loc[df['Attack'] != 'Benign', 'Attack'].value_counts()
    n_att = int((df['Attack'] != 'Benign').sum())
    return [f for f, c in v.items() if c >= min_rows and c / n_att <= max_share]

def metrics_at_threshold(y_true, p_pos, thr):
    """Threshold-sensitive metrics taken at an externally chosen cut rather than at 0.5.
    Needed because the rethreshold strategy does not operate at 0.5, so reporting its
    macro-F1 or false-positive rate from the default cut would describe a different detector."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred),
                macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
import numpy as np, pandas as pd

def point_biserial(X, y, chunk=200_000):
    """Per-feature correlation with the binary label. Computed in chunks so a 2M-row
    partition never needs a dense copy."""
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    n = len(y)
    sx = np.zeros(X.shape[1]); sxx = np.zeros(X.shape[1]); sxy = np.zeros(X.shape[1])
    sy = y.sum(); syy = (y * y).sum()
    for i in range(0, n, chunk):
        xb = X[i:i + chunk]; yb = y[i:i + chunk]
        sx += xb.sum(0); sxx += (xb * xb).sum(0); sxy += xb.T @ yb
    cov = sxy / n - (sx / n) * (sy / n)
    vx = np.maximum(sxx / n - (sx / n) ** 2, 0.0)
    vy = max(syy / n - (sy / n) ** 2, 1e-12)
    with np.errstate(invalid='ignore', divide='ignore'):
        r = cov / np.sqrt(vx * vy)
    return np.nan_to_num(r, nan=0.0)

def flipped_features(r_src, r_tgt, tau=0.05):
    """Features whose association with the label reverses sign between corpora, counting
    only those with non-negligible association on both sides so that noise near zero
    does not register as a flip."""
    both = (np.abs(r_src) >= tau) & (np.abs(r_tgt) >= tau)
    return np.where(both & (np.sign(r_src) != np.sign(r_tgt)))[0]

def sign_agreement(r_src, r_tgt, tau=0.05):
    """Source-importance-weighted fraction of features keeping their sign. This is the
    candidate predictor of whether a transferred ranking survives."""
    m = np.abs(r_src) >= tau
    if not m.any():
        return np.nan
    w = np.abs(r_src[m])
    agree = (np.sign(r_src[m]) == np.sign(r_tgt[m])).astype(float)
    return float((w * agree).sum() / w.sum())

def partial_holdout_buffer(train_full, family, frac, seed, keep_share, min_per_group=1):
    """Buffer in which one family is removed (keep_share=0) or retained at a reduced
    share. Comparing the two separates the effect of an unseen attack type from the
    effect of a less diverse buffer, since both buffers have the same total size."""
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in train_full.groupby('Attack', sort=True):
        share = k / len(train_full)
        n = int(round(len(g) * share * (keep_share if fam == family else 1.0)))
        n = min(len(g), max(min_per_group if (fam != family or keep_share > 0) else 0, n))
        if n > 0:
            parts.append(g.sample(n=n, random_state=seed))
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    # top up to k from families other than the held-out one, so size is comparable
    if len(out) < k:
        pool = train_full[(train_full['Attack'] != family) & (~train_full.index.isin(out.index))]
        if len(pool):
            out = pd.concat([out, pool.sample(n=min(k - len(out), len(pool)), random_state=seed)],
                            ignore_index=True)
    return out

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(V6_CSV, mode='a', index=False, header=not os.path.exists(V6_CSV))

def key(seed, src, tgt, m, b, s, extra=''):
    return (str(seed), src, tgt, m, f'{float(b):.6g}', s, str(extra))

done = set()
if os.path.exists(V6_CSV):
    prev = pd.read_csv(V6_CSV)
    done = set(key(r.seed, r.source, r.target, r.model, r.budget, r.strategy,
                   '' if pd.isna(r.held_family) else r.held_family) for r in prev.itertuples())
    print(f'resume: {len(done)} rows recorded')

def is_done(*k):
    return key(*k) in done

def mark(seed, src, tgt, m, b, s, metrics, n_train, extra='', **kw):
    record(dict(seed=seed, source=src, target=tgt, model=m, budget=b, strategy=s,
                n_train=n_train, held_family=extra, **metrics, **kw))
    done.add(key(seed, src, tgt, m, b, s, extra))
    print(f"  s{seed} {src}->{tgt} {m} b={b} {s}{('/'+str(extra)) if extra else ''}: "
          f"MCC={metrics.get('mcc', float('nan')):.3f}")

T0 = time.time()
def el():
    return f'[{(time.time()-T0)/60:5.1f}m]'

seed = CFG['seed']
parts = {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train_full=tr, train=stratified_cap(tr, CFG['train_cap'], seed),
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True))

# ---------- per-feature association, and the sign-agreement predictor ----------
if not os.path.exists(FEA_CSV):
    R = {}
    for tag in CFG['corpora']:
        d = parts[tag]['train_full']
        d = d.sample(n=min(CFG['corr_sample'], len(d)), random_state=seed)
        X, _ = clean_X(d, FEATURES)
        R[tag] = point_biserial(X.values, d['Label'].values)
        print(f'{el()} correlations {tag}: |r| max {np.abs(R[tag]).max():.3f}')
        del X, d; gc.collect()
    pd.DataFrame(R, index=FEATURES).to_csv(FEA_CSV)
    print('saved', FEA_CSV)
RFEAT = pd.read_csv(FEA_CSV, index_col=0)

PAIRS = [(s, t) for s in CFG['corpora'] for t in CFG['corpora'] if s != t]
FLIP = {}
rows = []
for s, t in PAIRS:
    rs, rt = RFEAT[s].values, RFEAT[t].values
    idx = flipped_features(rs, rt, CFG['tau'])
    FLIP[(s, t)] = idx
    rows.append(dict(source=s, target=t, n_flipped=len(idx),
                     agreement=sign_agreement(rs, rt, CFG['tau']),
                     flipped=';'.join(np.array(FEATURES)[idx][:8])))
FLIPTAB = pd.DataFrame(rows)
FLIPTAB.to_csv(f'{RESULT}/fc_inversion_features.csv', index=False)
print(FLIPTAB[['source','target','n_flipped','agreement']].round(3).to_string(index=False))

# ---------- causal test: drop the sign-reversed features, against a random-drop control ----------
for s, t in PAIRS:
    flip = FLIP[(s, t)]
    if len(flip) == 0:
        continue
    # zlib.crc32 is stable across processes; Python's hash() is salted per run, which
    # would make the random-drop control differ between sessions and break resume.
    rng = np.random.default_rng(zlib.crc32(f'{s}|{t}'.encode()))
    rand = rng.choice(np.setdiff1d(np.arange(len(FEATURES)), flip),
                      size=min(len(flip), len(FEATURES) - len(flip)), replace=False)
    for cond, drop in [('drop_flipped', flip), ('drop_random', rand)]:
        dropset = set(drop)
        keep = [f for i, f in enumerate(FEATURES) if i not in dropset]
        if all(is_done(seed, s, t, mname, 0, cond) for mname in MODELS):
            continue
        # the reduced design matrices depend on the pair and condition, not the model
        Xs, med = clean_X(parts[s]['train'], keep); ys = parts[s]['train']['Label'].values
        ev = parts[t]['eval']; yev = ev['Label'].values
        Xev, _ = clean_X(ev, keep, medians=med)
        for mname in MODELS:
            if is_done(seed, s, t, mname, 0, cond):
                continue
            mdl = make_model(mname, seed, len(Xs))
            t0 = time.time(); mdl.fit(Xs, ys); fs = round(time.time() - t0, 1)
            p = mdl.predict_proba(Xev)[:, 1]
            m = all_metrics(yev, p)
            bm, bt, bo = best_threshold_two_sided(yev, p)
            m['mcc_best_thr'] = bm
            mark(seed, s, t, mname, 0, cond, m, len(Xs), orient=bo,
                 n_features=len(keep), n_dropped=len(drop), fit_s=fs)
            del mdl; gc.collect()
        del Xs, Xev; gc.collect()

# ---------- durability control: family retained at a reduced share ----------
for tgt in CFG['corpora']:
    ev = parts[tgt]['eval']; yev = ev['Label'].values
    fams = eligible_families(parts[tgt]['train_full'], CFG['hold_min_rows'], CFG['hold_max_share'])[:CFG['max_families']]
    for fam in fams:
        buf = partial_holdout_buffer(parts[tgt]['train_full'], fam, CFG['hold_budget'], seed, CFG['hold_keep'])
        ybuf = buf['Label'].values
        if len(np.unique(ybuf)) < 2:
            continue
        Xb, mb = clean_X(buf, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
        for mname in MODELS:
            if is_done(seed, 'any', tgt, mname, CFG['hold_budget'], 'buffer_partial', fam):
                continue
            mdl = make_model(mname, seed, len(Xb))
            t0 = time.time(); mdl.fit(Xb, ybuf); fs = round(time.time() - t0, 1)
            p_ev = mdl.predict_proba(Xev_b)[:, 1]; p_bf = mdl.predict_proba(Xb)[:, 1]
            m = all_metrics(yev, p_ev)
            thr = threshold_for_fpr_on_buffer(ybuf, p_bf, CFG['fpr_caps'][0])
            hm = (ev['Attack'] == fam).values
            if hm.sum() and not np.isnan(thr):
                m['tpr_heldout_family'] = float((p_ev[hm] >= thr).mean())
                m['tpr_at_fpr01_buffer'] = realised_at_threshold(yev, p_ev, thr)[0]
            mark(seed, 'any', tgt, mname, CFG['hold_budget'], 'buffer_partial', m, len(buf),
                 extra=fam, keep_share=CFG['hold_keep'], fit_s=fs)
            del mdl; gc.collect()
        del Xb, Xev_b; gc.collect()

# ---------- replication of the update-size sweep ----------
for rseed in CFG['rep_seeds']:
    rparts = {}
    for tag, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=rseed)
        tr = tr.reset_index(drop=True)
        rparts[tag] = dict(train_full=tr, train=stratified_cap(tr, CFG['train_cap'], rseed),
                           eval=stratified_cap(te, CFG['eval_cap'], rseed).reset_index(drop=True))
    RBUF = {(tg, b): stratified_frac(rparts[tg]['train_full'], b, rseed)
            for tg in CFG['corpora'] for b in CFG['ft_budgets']}
    # the retraining comparator depends only on the target buffer, so it is fitted once
    # per (target, model, budget) rather than once per source as well
    for tgt in CFG['corpora']:
        ev = rparts[tgt]['eval']; yev = ev['Label'].values
        for b in CFG['ft_budgets']:
            buf = RBUF[(tgt, b)]; ybuf = buf['Label'].values
            if len(np.unique(ybuf)) < 2:
                continue
            if all(is_done(rseed, 'any', tgt, mname, b, 'buffer_only_rep') for mname in MODELS):
                continue
            Xb, mb = clean_X(buf, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
            for mname in MODELS:
                if is_done(rseed, 'any', tgt, mname, b, 'buffer_only_rep'):
                    continue
                bm = make_model(mname, rseed, len(Xb))
                t1 = time.time(); bm.fit(Xb, ybuf)
                m = all_metrics(yev, bm.predict_proba(Xev_b)[:, 1])
                mark(rseed, 'any', tgt, mname, b, 'buffer_only_rep', m, len(buf),
                     fit_s=round(time.time() - t1, 1))
                del bm; gc.collect()
            del Xb, Xev_b; gc.collect()

    for src in CFG['corpora']:
        others = [t for t in CFG['corpora'] if t != src]
        Xs, med_s = clean_X(rparts[src]['train'], FEATURES); ys = rparts[src]['train']['Label'].values
        for mname in MODELS:
            if all(is_done(rseed, src, t, mname, b, f'finetune_x{k}')
                   for t in others for b in CFG['ft_budgets'] for k in CFG['ft_mult']):
                continue
            src_model = make_model(mname, rseed, len(Xs))
            t0 = time.time(); src_model.fit(Xs, ys)
            print(f'{el()} seed {rseed} source fit {mname} on {src}: {time.time()-t0:.0f}s')
            for tgt in others:
                ev = rparts[tgt]['eval']; yev = ev['Label'].values
                Xev_s, _ = clean_X(ev, FEATURES, medians=med_s)
                for b in CFG['ft_budgets']:
                    buf = RBUF[(tgt, b)]; ybuf = buf['Label'].values
                    if len(np.unique(ybuf)) < 2:
                        continue
                    Xb_s, _ = clean_X(buf, FEATURES, medians=med_s)
                    pend = [k for k in CFG['ft_mult'] if not is_done(rseed, src, tgt, mname, b, f'finetune_x{k}')]
                    if pend:
                        ftm = copy.deepcopy(src_model); prev = 0
                        for k in CFG['ft_mult']:
                            step = (100 * k if mname in ('rf', 'lgbm') else 30 * k) - prev
                            prev += step
                            t1 = time.time()
                            ftm = finetune(ftm, mname, Xb_s, ybuf, rseed, add_trees=step, ft_epochs=step)
                            if k not in pend:
                                continue
                            m = all_metrics(yev, ftm.predict_proba(Xev_s)[:, 1])
                            mark(rseed, src, tgt, mname, b, f'finetune_x{k}', m, len(Xs) + len(buf),
                                 fit_s=round(time.time() - t1, 1))
                        del ftm; gc.collect()
                    del Xb_s
                del Xev_s; gc.collect()
            del src_model; gc.collect()
        del Xs; gc.collect()

print('rows recorded:', len(done))

In [ ]:
from scipy.stats import spearmanr, wilcoxon

v6 = pd.read_csv(V6_CSV).drop_duplicates(['seed','source','target','model','budget','strategy','held_family'])
v3 = pd.read_csv(f'{RESULT}/fc_results_v3.csv'); v5 = pd.read_csv(f'{RESULT}/fc_results_v5.csv')
v1 = pd.read_csv(f'{RESULT}/fc_results.csv')
FT = pd.read_csv(f'{RESULT}/fc_inversion_features.csv')

print('=== 1. DOES SIGN AGREEMENT PREDICT WHETHER THE RANKING SURVIVES? ===')
z = v3[v3.strategy=='zero_shot_2s'].groupby(['source','target']).agg(
        inverted=('orient', lambda s: float((s==-1).mean())), ceiling=('mcc_best_thr','mean')).reset_index()
J = z.merge(FT, on=['source','target'])
print(J[['source','target','n_flipped','agreement','inverted','ceiling']].round(3).to_string(index=False))
for col in ['agreement','n_flipped']:
    r1,p1 = spearmanr(J[col], J.inverted); r2,p2 = spearmanr(J[col], J.ceiling)
    print(f"  {col:11s} vs inversion rate rho={r1:+.3f} (p={p1:.3f}) | vs ceiling rho={r2:+.3f} (p={p2:.3f})")
J.round(4).to_csv(f'{RESULT}/fc_inversion_predictor.csv', index=False)

print('\n=== 2. CAUSAL TEST: removing sign-reversed features vs removing random features ===')
d = v6[v6.strategy.isin(['drop_flipped','drop_random'])]
base = v3[(v3.strategy=='zero_shot_2s') & (v3.seed==CFG['seed'])][['source','target','model','mcc','mcc_best_thr','orient']].rename(
        columns={'mcc':'base_mcc','mcc_best_thr':'base_ceiling','orient':'base_orient'})
D = d.pivot_table(index=['source','target','model'], columns='strategy',
                  values=['mcc','mcc_best_thr','orient']).reset_index()
D.columns = ['_'.join([c for c in col if c]).strip() for col in D.columns]
D = D.merge(base, on=['source','target','model'])
print(D[['mcc_drop_flipped','mcc_drop_random','base_mcc',
         'mcc_best_thr_drop_flipped','mcc_best_thr_drop_random','base_ceiling']].mean().round(3).to_string())
gain_f = D.mcc_drop_flipped - D.base_mcc; gain_r = D.mcc_drop_random - D.base_mcc
print(f"  zero-shot MCC change: dropping reversed features {gain_f.mean():+.3f}, dropping random {gain_r.mean():+.3f}, "
      f"difference p={wilcoxon(D.mcc_drop_flipped, D.mcc_drop_random).pvalue:.4f} (n={len(D)})")
inv_before = (D.base_orient==-1).mean(); inv_f = (D.orient_drop_flipped==-1).mean(); inv_r = (D.orient_drop_random==-1).mean()
print(f"  inverted cells: before {inv_before:.2f}, after dropping reversed {inv_f:.2f}, after dropping random {inv_r:.2f}")
D.round(4).to_csv(f'{RESULT}/fc_inversion_causal.csv', index=False)

print('\n=== 3. DURABILITY CONTROL: family absent vs retained at a tenth share ===')
h = pd.read_csv(f'{RESULT}/fc_family_holdout.csv')[['target','model','held_family','tpr_heldout_family','mcc']].rename(
        columns={'tpr_heldout_family':'tpr_absent','mcc':'mcc_absent'})
p_ = v6[v6.strategy=='buffer_partial'][['target','model','held_family','tpr_heldout_family','mcc']].rename(
        columns={'tpr_heldout_family':'tpr_tenth','mcc':'mcc_tenth'})
C = h.merge(p_, on=['target','model','held_family'])
print(C.groupby('target')[['tpr_absent','tpr_tenth','mcc_absent','mcc_tenth']].mean().round(3).to_string())
print(f"  overall recall on the family: absent {C.tpr_absent.mean():.3f}, tenth share {C.tpr_tenth.mean():.3f}, "
      f"p={wilcoxon(C.tpr_absent, C.tpr_tenth).pvalue:.4f} (n={len(C)})")
C.round(4).to_csv(f'{RESULT}/fc_durability_control.csv', index=False)

print('\n=== 4. UPDATE-SIZE SWEEP ACROSS THREE SEEDS ===')
ft5 = v5[v5.strategy.str.startswith('finetune_x')][['source','target','model','budget','strategy','mcc']].assign(seed=CFG['seed'])
ft6 = v6[v6.strategy.str.startswith('finetune_x')][['seed','source','target','model','budget','strategy','mcc']]
FTA = pd.concat([ft5, ft6]); FTA['mult'] = FTA.strategy.str.replace('finetune_x','').astype(int)
bo5 = v1[(v1.strategy=='buffer_only')&(v1.seed==CFG['seed'])][['source','target','model','budget','mcc']].assign(seed=CFG['seed'])
bo6 = v6[v6.strategy=='buffer_only_rep'][['seed','target','model','budget','mcc']]
BO = pd.concat([bo5.drop(columns=['source']), bo6]).rename(columns={'mcc':'bo'})
M = FTA.pivot_table(index=['seed','source','target','model','budget'], columns='mult', values='mcc').reset_index().merge(
        BO, on=['seed','target','model','budget'])
print(M.groupby('model')[[1,3,10,'bo']].agg(['mean','std']).round(3).to_string())
print('\n  by seed:'); print(M.groupby('seed')[[1,3,10,'bo']].mean().round(3).to_string())
for k in [1,3,10]:
    d_ = M['bo'] - M[k]; pair = M.groupby(['source','target'])[['bo',k]].mean()
    print(f"  x{k}: buffer_only leads {d_.mean():+.3f}, wins {int((d_>0).sum())}/{len(d_)}, pair p={wilcoxon(pair.bo,pair[k]).pvalue:.4f}")
rf = M[M.model=='rf']; pair = rf.groupby(['source','target'])[['bo',10]].mean()
print(f"  RF at x10: finetune {rf[10].mean():.3f} vs buffer_only {rf.bo.mean():.3f}, "
      f"diff {(rf.bo-rf[10]).mean():+.3f}, pair p={wilcoxon(pair.bo,pair[10]).pvalue:.3f}, n={len(rf)}")
M.round(4).to_csv(f'{RESULT}/fc_finetune_sweep_3seed.csv', index=False)

print('\n=== 5. PER-CORPUS REPORTING (pooling across prevalence) ===')
bo_all = v1[v1.strategy=='buffer_only'].groupby(['target','budget']).mcc.mean().unstack().round(3)
print('buffer-only MCC by target and budget:'); print(bo_all.to_string())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "15: inversion mechanism (feature sign-reversal, causal drop test), durability control, update-sweep replication at 3 seeds"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)